# Limpeza de Dados — Diagnóstico Inicial

Este notebook **não limpa nada ainda**. O objetivo é apenas enxergar os problemas existentes no arquivo `dados/vendas.csv` antes de qualquer tratamento.

## 1. Carregar os dados

In [ ]:
import pandas as pd

# O parâmetro sep=',' indica que o separador de colunas é vírgula (padrão de CSV)
# dtype=str carrega tudo como texto por enquanto — assim nada é convertido automaticamente
# e conseguimos ver os valores exatamente como estão no arquivo
df = pd.read_csv('../dados/vendas.csv', dtype=str)

print(f"Linhas: {df.shape[0]}  |  Colunas: {df.shape[1]}")


## 2. Primeiras linhas

`df.head()` mostra as 5 primeiras linhas — serve para ter uma ideia rápida da estrutura e dos valores.

In [ ]:
df.head(10)


## 3. Visão geral das colunas

`df.info()` mostra o tipo de cada coluna e quantos valores não-nulos existem em cada uma.  
Como carregamos tudo como `str`, todas as colunas aparecem como `object` — o importante aqui é ver se alguma coluna tem menos de 100 entradas (o que indicaria valores faltantes).

In [ ]:
df.info()


## 4. Valores faltantes por coluna

`df.isnull().sum()` conta quantas células estão vazias em cada coluna.  
Células vazias no CSV viram `NaN` (Not a Number) no pandas — é assim que ele representa "ausência de valor".

In [ ]:
valores_faltantes = df.isnull().sum()

# Filtra só as colunas que têm pelo menos 1 valor faltante
print("Colunas com valores faltantes:")
print(valores_faltantes[valores_faltantes > 0])


In [ ]:
# Versão visual: mostra todas as colunas com percentual de dados faltantes
percentual = (df.isnull().sum() / len(df) * 100).round(1)
resumo_faltantes = pd.DataFrame({
    'qtd_faltantes': df.isnull().sum(),
    'percentual (%)': percentual
})

resumo_faltantes[resumo_faltantes['qtd_faltantes'] > 0]


## 5. Problema de capitalização — coluna `produto`

O mesmo produto aparece escrito de formas diferentes: `"notebook"`, `"Notebook"`, `"NOTEBOOK"`.  
Para o pandas, esses são três produtos distintos — o que vai estragar qualquer análise de vendas por produto.

`df['produto'].unique()` mostra todos os valores únicos da coluna.

In [ ]:
produtos_unicos = sorted(df['produto'].dropna().unique())

print(f"Total de valores únicos em 'produto': {len(produtos_unicos)}")
print()
for p in produtos_unicos:
    print(f"  '{p}'")


In [ ]:
# Consequência do problema: o pandas enxerga 18 "produtos" em vez de 6
# Se tentarmos contar vendas por produto agora, o resultado vai ser errado
print("Contagem de vendas por produto (ANTES da limpeza — resultado incorreto):")
print(df['produto'].value_counts())


## 6. Problema de formato — coluna `data_venda`

As datas estão em três formatos diferentes no mesmo arquivo:
- `2024-01-03` — formato ISO (ano-mês-dia)
- `05/01/2024` — formato brasileiro com barra (dia/mês/ano)
- `09-01-2024` — formato com hífen (dia-mês-ano)

Enquanto não padronizarmos, o pandas não consegue fazer operações com datas (ordenar, filtrar por mês, calcular períodos etc.).

In [ ]:
datas_unicas = sorted(df['data_venda'].dropna().unique())

print(f"Total de datas únicas: {len(datas_unicas)}")
print()
for d in datas_unicas:
    print(f"  '{d}'")


In [ ]:
# Identifica qual formato cada data usa
import re

formato_iso    = []  # 2024-01-03
formato_barra  = []  # 05/01/2024
formato_hifen  = []  # 09-01-2024

for data in datas_unicas:
    if re.match(r'^\d{4}-\d{2}-\d{2}$', data):
        formato_iso.append(data)
    elif re.match(r'^\d{2}/\d{2}/\d{4}$', data):
        formato_barra.append(data)
    elif re.match(r'^\d{2}-\d{2}-\d{4}$', data):
        formato_hifen.append(data)

print(f"Formato ISO  (YYYY-MM-DD):  {len(formato_iso)} datas   → ex: {formato_iso[0]}")
print(f"Formato barra (DD/MM/YYYY): {len(formato_barra)} datas   → ex: {formato_barra[0]}")
print(f"Formato hífen (DD-MM-YYYY): {len(formato_hifen)} datas   → ex: {formato_hifen[0]}")


## 7. Problema de formato — coluna `preco_unitario`

Os preços usam vírgula como separador decimal e ponto como separador de milhar (padrão brasileiro).  
Ex: `"3.500,00"` em vez de `3500.00`.

O pandas não consegue converter esse formato diretamente para número — precisa de tratamento antes.

In [ ]:
print("Valores únicos em 'preco_unitario':")
print(df['preco_unitario'].unique())
print()

# Tenta converter para número — vai falhar por causa da vírgula
try:
    pd.to_numeric(df['preco_unitario'])
except Exception as erro:
    print(f"Erro ao converter para número: {erro}")


## 8. Resumo dos problemas encontrados

| Problema | Coluna(s) | Impacto |
|---|---|---|
| Valores faltantes | `quantidade`, `vendedor`, `preco_unitario` | Cálculos de total ficam errados ou incompletos |
| Capitalização inconsistente | `produto` | Agrupa produtos iguais como diferentes |
| Formatos de data misturados | `data_venda` | Não consegue ordenar nem filtrar por período |
| Separador decimal com vírgula | `preco_unitario` | Não consegue fazer cálculos numéricos |

**Próximo passo:** limpar cada um desses problemas nas seções abaixo.

---

# Limpeza dos Dados

Agora que sabemos exatamente quais são os problemas, vamos corrigi-los um por um.  
Todas as alterações são feitas em cima do `df` original — o arquivo `vendas.csv` nunca é tocado.

## 9. Padronizar a coluna `produto` (title case)

**Title case** significa capitalizar a primeira letra de cada palavra: `"cadeira gamer"` → `"Cadeira Gamer"`.  
O método `.str.title()` do pandas faz isso em uma linha para a coluna inteira.

In [ ]:
# .str acessa métodos de texto para colunas do pandas
# .title() transforma a string para title case: primeira letra de cada palavra em maiúscula
df['produto'] = df['produto'].str.title()

# Verifica o resultado: agora deve haver apenas 6 produtos únicos
print("Produtos únicos após padronização:")
print(sorted(df['produto'].dropna().unique()))
print()
print("Contagem de vendas por produto (agora correta):")
print(df['produto'].value_counts())


## 10. Unificar os formatos de data para `YYYY-MM-DD`

O pandas tem a função `pd.to_datetime()` que converte strings para datas.  
O problema aqui é que temos **três formatos diferentes no mesmo arquivo**, então não podemos passar um único formato.

A solução é processar cada formato separadamente e depois juntar tudo:
1. Datas ISO (`2024-01-03`) já estão no formato certo — só converter
2. Datas com barra (`05/01/2024`) — informamos o formato `%d/%m/%Y`
3. Datas com hífen (`09-01-2024`) — informamos o formato `%d-%m-%Y`

In [ ]:
def converter_data(data):
    """Recebe uma string de data em qualquer um dos três formatos e retorna um objeto datetime."""
    if pd.isnull(data):
        return pd.NaT  # NaT é o equivalente de NaN para datas

    # Tenta cada formato em ordem — o primeiro que funcionar é retornado
    for formato in ('%Y-%m-%d', '%d/%m/%Y', '%d-%m-%Y'):
        try:
            return pd.to_datetime(data, format=formato)
        except ValueError:
            continue  # esse formato não funcionou, tenta o próximo

    # Se nenhum formato funcionou, retorna NaT para não quebrar o código
    return pd.NaT

# Aplica a função em cada linha da coluna data_venda
# .apply() percorre a coluna linha por linha e chama a função em cada valor
df['data_venda'] = df['data_venda'].apply(converter_data)

# Verifica o resultado
print(f"Tipo da coluna depois da conversão: {df['data_venda'].dtype}")
print()
print("Primeiras datas convertidas:")
print(df['data_venda'].head(10))


## 11. Converter `preco_unitario` de string com vírgula para float

Os preços estão no formato brasileiro: `"3.500,00"` (ponto = milhar, vírgula = decimal).  
Para o Python, o formato numérico correto é `3500.00` (sem ponto de milhar, ponto = decimal).

Precisamos fazer dois ajustes na string antes de converter:
1. Remover o ponto de milhar: `"3.500,00"` → `"3500,00"`
2. Trocar a vírgula pelo ponto: `"3500,00"` → `"3500.00"`

Depois disso, `pd.to_numeric()` consegue converter normalmente.

In [ ]:
# Passo 1: remove o ponto de milhar  → "3.500,00" vira "3500,00"
# Passo 2: troca a vírgula pelo ponto → "3500,00"  vira "3500.00"
# .str.replace() substitui texto dentro de cada célula da coluna
df['preco_unitario'] = (
    df['preco_unitario']
    .str.replace('.', '', regex=False)   # remove o ponto (regex=False = trata como texto simples)
    .str.replace(',', '.', regex=False)  # troca vírgula por ponto
)

# Agora converte para número de ponto flutuante (float)
# errors='coerce' transforma qualquer valor que não conseguir converter em NaN
# (em vez de quebrar com erro)
df['preco_unitario'] = pd.to_numeric(df['preco_unitario'], errors='coerce')

# Verifica o resultado
print(f"Tipo da coluna depois da conversão: {df['preco_unitario'].dtype}")
print()
print("Valores únicos de preco_unitario:")
print(sorted(df['preco_unitario'].dropna().unique()))


## 12. Tratar valores faltantes

Temos três situações distintas que pedem estratégias diferentes:

- **`quantidade`** — valor numérico. Usar a **mediana do produto** é mais inteligente do que a mediana geral, porque produtos diferentes têm padrões de compra diferentes (ninguém compra 2 notebooks de uma vez, mas pode comprar 5 mouses).
- **`preco_unitario`** — também numérico, com a mesma lógica: cada produto tem seu próprio preço, então faz mais sentido usar a mediana por produto do que um valor geral.
- **`vendedor`** — não tem como estimar quem fez a venda. A opção honesta é preencher com `"Não informado"` para deixar claro que o dado está ausente.

### O que é mediana?
A mediana é o valor do meio quando você ordena todos os números.  
Ela é preferível à média quando existem valores muito discrepantes (ex: uma venda de 10 notebooks distorceria a média).


In [ ]:
# Antes de preencher: quantidade ainda está como string — precisa converter para número
df['quantidade'] = pd.to_numeric(df['quantidade'], errors='coerce')

print("Valores faltantes ANTES do preenchimento:")
print(df[['quantidade', 'preco_unitario', 'vendedor']].isnull().sum())


In [ ]:
# --- quantidade: preenche com a mediana do produto correspondente ---

# groupby('produto') agrupa as linhas por produto
# transform('median') calcula a mediana de cada grupo e devolve uma Series
# com o mesmo tamanho do df original — cada linha recebe a mediana do seu produto
mediana_quantidade_por_produto = df.groupby('produto')['quantidade'].transform('median')

# fillna() preenche apenas os NaN, mantendo os valores existentes
# .astype(int) converte para inteiro — só é seguro fazer isso depois do fillna(),
# porque int não aceita NaN (por isso a coluna vira float quando há nulos)
df['quantidade'] = df['quantidade'].fillna(mediana_quantidade_por_produto).astype(int)

# --- preco_unitario: mesma lógica ---
mediana_preco_por_produto = df.groupby('produto')['preco_unitario'].transform('median')
df['preco_unitario'] = df['preco_unitario'].fillna(mediana_preco_por_produto)

# --- vendedor: preenche com texto fixo ---
df['vendedor'] = df['vendedor'].fillna('Não informado')

print("Valores faltantes DEPOIS do preenchimento:")
print(df[['quantidade', 'preco_unitario', 'vendedor']].isnull().sum())
print()
print(f"Tipo da coluna quantidade: {df['quantidade'].dtype}")
print(df['quantidade'].head(10))


In [ ]:
# Confere quais linhas tinham vendedor faltante — agora com "Não informado"
print("Linhas onde vendedor foi preenchido automaticamente:")
df[df['vendedor'] == 'Não informado'][['id_venda', 'produto', 'vendedor']]


## 13. Verificação final antes de salvar

Antes de gravar o arquivo limpo, confere o estado do `df` como um todo.

In [ ]:
print("=== Visão geral do dataframe limpo ===")
print()
df.info()


In [ ]:
print("Valores faltantes restantes:")
print(df.isnull().sum())
print()
print("Primeiras linhas do dataframe limpo:")
df.head(10)


## 14. Salvar o arquivo limpo

Salva o resultado em `dados/vendas_limpo.csv`.  
O arquivo original `dados/vendas.csv` permanece intacto — boa prática em projetos de dados.

In [ ]:
# index=False evita que o pandas escreva a coluna de índice (0, 1, 2...) no CSV
# date_format='%Y-%m-%d' garante que as datas sejam salvas no formato ISO, sem hora
df.to_csv('../dados/vendas_limpo.csv', index=False, date_format='%Y-%m-%d')

print("Arquivo salvo em: dados/vendas_limpo.csv")
print(f"Linhas salvas: {len(df)}")
